# Занятие 14. Практика: кластеризация на простых данных

На теоретическом занятии разобраны понятия: **признак**, **расстояние**, **центроид**, **K-Means**,
**иерархическая кластеризация**, **дендрограмма**, **метод локтя**. Здесь эти методы применяются
на данных из двух разных областей — образование и медицина.

**План занятия:**

1. Разбор **примера** — группировка студентов по пройденным модулям курса.
2. **Практика**

В каждой ячейке практики после комментария `# Ваш код` пишется решение; где просят вывод —
он оформляется в ячейке `**Вывод:**` под кодом.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

plt.rcParams['figure.dpi'] = 90
plt.rcParams['font.family'] = 'DejaVu Sans'
%matplotlib inline

---
## Эталонный пример. Пройденные модули курса

Набор `modules.csv` — **34 студента** онлайн-курса и **8 модулей**. Для каждого студента указано
`1` (модуль пройден) или `0` (не пройден):

| Модуль | Тема | Модуль | Тема |
|---|---|---|---|
| HTML | вёрстка страниц | PYTHON | программирование на Python |
| CSS | стили и оформление | STATS | статистика и анализ |
| JS | JavaScript | FIGMA | макеты в Figma |
| SQL | базы данных | UX | проектирование интерфейсов |

Признаков восемь — на плоскости не нарисовать. Но порядок действий тот же, что и в учебном примере
занятия: дендрограмма → каменистая осыпь → разрез дерева → центроиды → сравнение с K-Means.

### Шаг 1. Загрузка и просмотр

In [ ]:
df = pd.read_csv('data/modules.csv')
print('Размер таблицы:', df.shape)
df.head()

Оценим **популярность** каждого модуля — среднее по столбцу (доля прошедших).

In [ ]:
df.mean().sort_values(ascending=False).round(2)

**Наблюдение.** Чаще всего проходят `HTML` (0.50), `JS` (0.44), `CSS`/`FIGMA`/`UX` (0.41);
реже — `SQL` и `STATS`. Похоже, группы соберутся вокруг трёх учебных треков: веб, данные, дизайн.

### Шаг 2. Иерархическая кластеризация и дендрограмма
`linkage` c `method='ward'`, `metric='euclidean'`; `dendrogram` рисует дерево склеек.

In [ ]:
link = linkage(df, method='ward', metric='euclidean')

plt.figure(figsize=(12, 5))
dendrogram(link)
plt.title('Дендрограмма: модули (ward, euclidean)')
plt.xlabel('Номер студента'); plt.ylabel('Расстояние при слиянии')
plt.show()

**Наблюдение.** Видны **3 крупные ветви** — три устойчивые группы студентов.

### Шаг 3. Число групп: каменистая осыпь
Смотрим расстояния слияний в обратном порядке и ищем точку, после которой график перестаёт резко падать.

In [ ]:
dist_rev = link[:, 2][::-1]
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(dist_rev) + 1), dist_rev, marker='o')
plt.title('Каменистая осыпь (модули)')
plt.xlabel('Число кластеров'); plt.ylabel('Расстояние при слиянии')
plt.xlim(0, 12); plt.grid(True, alpha=0.3)
plt.show()
print('Первые расстояния:', [round(x, 1) for x in dist_rev[:5]])

**Наблюдение.** Резкое падение до 3 кластеров (7.6 → 5.2 → 2.5), дальше линия почти плоская.
Точка перелома — `K = 3`.

### Шаг 4. Разрез дерева — `fcluster`

In [ ]:
df['cluster'] = fcluster(link, 3, criterion='maxclust')
df.groupby('cluster').size()

**Наблюдение.** Группы получились не вырожденными (примерно 11 / 12 / 11 студентов) — разбиение осмысленно.

### Шаг 5. Профили групп (центроиды)
Для бинарных данных `groupby().mean()` — это **доля прошедших модуль** внутри группы:
близко к 1 — модуль характерен для группы, близко к 0 — нет.

In [ ]:
df.groupby('cluster').mean().round(2)

**Наименование групп по профилям:**

- **Data-трек** — высокие `SQL`, `PYTHON`, `STATS`, веб- и дизайн-модули ≈ 0.
- **Дизайн-трек** — высокие `UX` и `FIGMA`, немного `HTML`.
- **Веб-трек** — высокие `HTML`, `CSS`, `JS`.

### Шаг 6. K-Means и сравнение методов
Применяем второй метод и через `crosstab` проверяем совпадение групп с иерархией.

In [ ]:
X = df.drop(columns='cluster')
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df['kmeans'] = km.fit_predict(X)
pd.crosstab(df['cluster'], df['kmeans'])

**Наблюдение.** Номера групп у двух методов свои, поэтому главная диагональ здесь ни при чём —
смотреть надо на структуру таблицы. Каждый иерархический кластер почти целиком укладывается в **один**
столбец K-Means: кластеры 1 и 3 (по 11 объектов) совпали полностью, а у кластера 2 из его 12 объектов
11 попали в один столбец и лишь **один** — в другой. Итого расхождение — **один объект из 34**:
методы выделили практически одни и те же группы, и эта согласованность подтверждает разбиение.

**Вывод по эталонному примеру.** Один и тот же порядок действий — дендрограмма, каменистая осыпь,
`fcluster`, центроиды, сравнение с K-Means — позволил выделить и содержательно описать три учебных
трека. Далее эти шаги выполняются самостоятельно.

---
## Практика

Ниже **10 заданий** на двух наборах данных, всего **30 баллов**.

- **Задания 1–6** — набор `students.csv` (навыки студентов).
- **Задания 7–10** — набор `health.csv` (показатели здоровья пациентов).

В каждой ячейке после `# Ваш код` напишите решение. Где просят прокомментировать результат —
заполните ячейку `**Вывод:**` под кодом.

Параметры расчётов везде одинаковы: `linkage(method='ward', metric='euclidean')`,
`fcluster(..., criterion='maxclust')`, `KMeans(..., random_state=42, n_init=10)`.

### Знакомство с датасетами практики

Перед решением заданий посмотрим на оба набора: сколько строк и столбцов, как выглядят первые записи
и какие в них значения.

**Набор 1 — навыки студентов** (`students.csv`). Столбец `NAME` — имя студента, `S1–S10` — баллы навыков.

In [ ]:
students_preview = pd.read_csv('data/students.csv')
print('Размер:', students_preview.shape)
students_preview.head()

**Набор 2 — показатели здоровья** (`health.csv`) — шесть медицинских показателей.

In [ ]:
health_preview = pd.read_csv('data/health.csv')
print('Размер:', health_preview.shape)
health_preview.head()

Посмотрим на разброс показателей здоровья — минимум, максимум и среднее по каждому столбцу.

In [ ]:
health_preview.describe().round(1)

**Наблюдение.** Показатели живут в **разных диапазонах**: `pulse` — десятки (~60-90),
а `chol` — сотни (~150-350). Из-за этого «крупные» признаки будут доминировать в расстоянии —
поэтому в задании 7 данные придётся **стандартизовать**.

### Набор 1. Навыки студентов

18 студентов прошли оценку по 10 навыкам (баллы 1–10). Навыки **S1–S4** — аналитические
(S1 логика, S2 математика, S3 алгоритмы, S4 анализ данных), **S5–S10** — гуманитарные и
коммуникативные (S5 письменная речь, S6 презентация, S7 работа в команде, S8 иностранный язык,
S9 креативность, S10 эмпатия).

### <font color='DarkOrange'>Задание 1 [2 балла]</font>

Загрузите `data/students.csv`. Сохраните список названий навыков `S1–S10` в переменную `skills`.
Выведите размер таблицы (`.shape`).

In [ ]:
# Ваш код



### <font color='DarkOrange'>Задание 2 [3 балла]</font>

Постройте иерархическую кластеризацию по навыкам `S1–S10`
(`linkage`, `method='ward'`, `metric='euclidean'`) и дендрограмму с именами студентов
(`labels=stu['NAME'].tolist()`).

In [ ]:
# Ваш код



**Вывод:** на дендрограмме выделяются **3 крупные ветви** — три устойчивые группы студентов.

### <font color='DarkOrange'>Задание 3 [3 балла]</font>

Постройте каменистую осыпь (расстояния слияний в обратном порядке) и по ней определите
подходящее число групп.

In [ ]:
# Ваш код



**Вывод:** расстояния резко падают до 3 групп (32.9 → 20.9 → 10.8), дальше почти плоско —
естественное число групп **K = 3**; при детализации осмысленно и **K = 4**.

### <font color='DarkOrange'>Задание 4 [3 балла]</font>

Разрежьте дерево (`fcluster`, `criterion='maxclust'`) на **3** и на **4** кластера.
Для каждого разбиения выведите размеры групп.

In [ ]:
# Ваш код



### <font color='DarkOrange'>Задание 5 [3 балла]</font>

Для разбиения на **4** кластера вычислите центроиды — средние баллы по навыкам `S1–S10`
(`groupby('cl4')[skills].mean()`, округлите до 1 знака). По профилям **содержательно опишите группы**.

In [ ]:
# Ваш код



**Вывод:** четыре группы студентов по профилям центроидов:
- **Аналитики** — высокие S1–S4 (~9), низкие гуманитарные S5–S10 (~4).
- **Универсалы** — высокие баллы почти по всем навыкам (~9).
- **Гуманитарии со средней аналитикой** — умеренные S1–S4 (~6), высокие S5–S10 (~8).
- **Гуманитарии** — низкие S1–S4 (~4), высокие S5–S10 (~8.7).

### <font color='DarkOrange'>Задание 6 [3 балла]</font>

Выполните `KMeans` на **4** кластера (`random_state=42`, `n_init=10`) по навыкам `S1–S10`
и сравните с иерархическим разбиением `cl4` через `pd.crosstab`. Согласуются ли методы?

In [ ]:
# Ваш код



**Вывод:** номера групп у методов свои, но каждый иерархический кластер **целиком** совпал с одной
группой K-Means (в каждой строке заполнен ровно один столбец, без расхождений). Иерархия и K-Means
выделили **одни и те же** 4 группы — разбиение устойчиво.

### Набор 2. Показатели здоровья пациентов

Файл `health.csv` — **310 пациентов** и 6 медицинских показателей:

| Показатель | Что означает | Показатель | Что означает |
|---|---|---|---|
| `age` | возраст, годы | `sbp` | систолическое давление, мм рт.ст. |
| `bmi` | индекс массы тела | `pulse` | пульс покоя, уд/мин |
| `glucose` | глюкоза натощак, мг/дл | `chol` | холестерин, мг/дл |

Показатели измерены в **разных шкалах** (пульс ~60-90, а холестерин ~150-350).
Задача — выделить **группы риска** сердечно-сосудистых заболеваний.

### <font color='DarkOrange'>Задание 7 [3 балла]</font>

Загрузите `data/health.csv`. Стандартизуйте все 6 показателей
через `StandardScaler` (результат — в `DataFrame` с теми же названиями столбцов).
Выведите размер набора.

In [ ]:
# Ваш код



### <font color='DarkOrange'>Задание 8 [3 балла]</font>

На **стандартизованных** данных постройте иерархическую кластеризацию
(`method='ward'`, `metric='euclidean'`) и каменистую осыпь. Определите число групп риска.
Дендрограмму стройте с `truncate_mode='lastp', p=20` (объектов много).

In [ ]:
# Ваш код



**Вывод:** расстояния резко падают до 3 групп (42.7 -> 23.2 -> 13.0), дальше линия почти плоская.
Число групп риска **K = 3**.

### <font color='DarkOrange'>Задание 9 [4 балла]</font>

Разрежьте дерево на **3** группы (`fcluster`), выведите размеры групп и центроиды
(средние показатели по группам, округление до 1). **Содержательно опишите** каждую группу
как уровень риска.

In [ ]:
# Ваш код



**Вывод:** три группы риска (размеры ~ 133 / 107 / 70):
- **Низкий риск** (~133 чел.) — молодые (возраст ~45), нормальный `bmi` (~22), глюкоза, давление и пульс в норме.
- **Умеренный риск** (~107 чел.) — средние значения по всем показателям (возраст ~55, `bmi` ~27, `sbp` ~135).
- **Высокий риск** (~70 чел.) — возраст ~66, `bmi` ~33, повышенные `glucose` (~145), `sbp` (~159), `pulse` (~80).

Холестерин `chol` почти одинаков во всех группах (~205-210) — сам по себе он группы риска **не различает**.

### <font color='DarkOrange'>Задание 10 [3 балла]</font>

Выполните иерархическую кластеризацию на **исходных** (нестандартизованных) показателях
на 3 группы и сравните разбиение со стандартизованным (`cluster`) через `pd.crosstab`.
Объясните результат.

In [ ]:
# Ваш код



**Вывод:** без стандартизации разбиение **заметно отличается** — в `crosstab` объекты
размазаны сразу по нескольким ячейкам в каждой строке, а не собираются по одному столбцу (нет
одно-однозначного соответствия групп). Причина: `chol` и `glucose` измерены в больших числах,
и в евклидовом расстоянии доминирует холестерин (у него самый большой разброс), хотя группы риска
он не различает. Стандартизация уравнивает вклад всех показателей — только тогда выделяются
корректные группы риска. **На данных с разными шкалами стандартизация обязательна.**